# 🕸️ DeepGuard — Notebook 3: Graph Neural Network (Multi-GNN)
**IBM AML Dataset** | Iqra University FYP 2023 | Supervisor: Dr. Dure e Jabeen

> ⚠️ **Read this before running.** Unlike Notebooks 1–2, this notebook was **not executed and
> verified** before being handed to you — the sandbox that built it couldn't install PyTorch
> (disk-quota limits unrelated to your Colab environment). The logic follows a published,
> peer-reviewed method as closely as reasonably possible for an FYP timeline, but you are the
> first person actually running this code. Expect to debug it. If you hit an error you can't
> resolve, paste it back and we'll fix it together — that's normal for this kind of work, not a
> sign something is fundamentally wrong.
>
> **Why bother, given that:** the published benchmark this follows reports **F1 ≈ 0.71, AP ≈ 0.67**
> on this exact dataset family — a real improvement over the XGBoost result (F1 0.62, AP 0.58) in
> Notebook 3. It's also a better fit for your thesis narrative: DeepGuard visualizes transactions
> as a network graph (Cytoscape.js), so a model that actually *reasons* over that graph structure
> is more defensible than a tabular model with a graph UI bolted on.
>
> **What this implements**, based on Egressy et al., *"Provably Powerful Graph Neural Networks
> for Directed Multigraphs"* (the paper that benchmarks GNNs on this exact IBM AML dataset family):
> 1. **Reverse message passing** — each transaction edge is duplicated in reverse with a
>    direction flag, so information flows both ways along a payment (the single biggest lever in
>    the paper's ablation: +28 F1 points on its own).
> 2. **Port numbering** — an edge feature counting repeated transactions between the same pair of
>    accounts, which helps the model recognize structuring/smurfing patterns (many small transfers
>    between the same accounts).
>
> Run **Notebook 1** first, or at least have `HI-Small_Trans.csv` ready to upload below.
>
> ⚡ **Use a GPU runtime**: Runtime → Change runtime type → T4 GPU. This will be painfully slow
> on CPU.

## 📦 Step 1 — Install PyTorch Geometric

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}  CUDA available: {torch.cuda.is_available()}")

!pip install torch_geometric -q
import torch_geometric
print(f"PyTorch Geometric: {torch_geometric.__version__}")

# NOTE: this deliberately avoids the optional pyg-lib/torch-scatter/torch-sparse compiled
# extensions (they need a version-matched wheel from data.pyg.org and are a common source of
# install headaches). GINEConv, used below, does not require them.

## 📂 Step 2 — Load the raw CSV

In [ ]:
import pandas as pd, numpy as np

RAW_CSV_PATH = '/content/HI-Small_Trans.csv'   # 🔧 change if needed

# if 'google.colab' in str(get_ipython()):
#     from google.colab import files
#     print("📁 Upload HI-Small_Trans.csv")
#     uploaded = files.upload()
#     RAW_CSV_PATH = f'/content/{list(uploaded.keys())[0]}'

df = pd.read_csv(RAW_CSV_PATH)
print(f"📊 Loaded: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")
print(f"🏷️  Fraud rate: {df['Is Laundering'].mean()*100:.4f}%  ({df['Is Laundering'].sum():,} / {len(df):,})")
df.head()

## 🕸️ Step 3 — Build the transaction graph

- **Nodes** = accounts (the `Account` / `Account.1` columns — already globally unique in this
  dataset, per Notebook 1's preprocessing)
- **Edges** = transactions, directed `Account → Account.1`
- **Edge features** = the same transaction-level features as Notebook 1 (amounts, time,
  currency/format), **plus** the two GNN-specific additions below
- **Node features** = simple degree/volume statistics computed from **training edges only**, to
  avoid leaking test-set information into what the model "knows" about a node

In [ ]:
from sklearn.preprocessing import LabelEncoder

df = df.dropna(subset=['Timestamp', 'Amount Paid', 'Amount Received', 'Account', 'Account.1']).reset_index(drop=True)

df['Timestamp']  = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Hour']       = df['Timestamp'].dt.hour
df['DayOfWeek']  = df['Timestamp'].dt.dayofweek
df['IsWeekend']  = (df['DayOfWeek'] >= 5).astype(int)
df['IsNightTx']  = ((df['Hour'] >= 22) | (df['Hour'] <= 5)).astype(int)

df['Amount Paid']     = pd.to_numeric(df['Amount Paid'], errors='coerce').fillna(0)
df['Amount Received'] = pd.to_numeric(df['Amount Received'], errors='coerce').fillna(0)
df['Log_Amount_Paid'] = np.log1p(df['Amount Paid'])
df['Amount_Diff']     = df['Amount Paid'] - df['Amount Received']

le_fmt = LabelEncoder(); df['Payment Format_enc'] = le_fmt.fit_transform(df['Payment Format'].astype(str))
le_cur = LabelEncoder(); df['Currency_enc'] = le_cur.fit_transform(df['Payment Currency'].astype(str))

# ── Chronological ordering matters for port numbering (defined as "the Nth transaction
# between this pair, in time order") ──
df = df.sort_values('Timestamp').reset_index(drop=True)

# ── Port numbering: count of prior transactions between this exact (sender, receiver) pair ──
df['port_number'] = df.groupby(['Account', 'Account.1']).cumcount()
print(f"✅ Port numbering: max repeated transactions between one pair = {df['port_number'].max()}")

# ── Global account → node-index mapping ──
all_accounts = pd.unique(pd.concat([df['Account'], df['Account.1']]))
acct_to_idx = {acct: i for i, acct in enumerate(all_accounts)}
n_nodes = len(all_accounts)
print(f"✅ {n_nodes:,} unique accounts (nodes)")

df['src'] = df['Account'].map(acct_to_idx)
df['dst'] = df['Account.1'].map(acct_to_idx)

In [ ]:
EDGE_FEATURE_COLS = [
    'Amount Paid', 'Amount Received', 'Log_Amount_Paid', 'Amount_Diff',
    'Hour', 'DayOfWeek', 'IsWeekend', 'IsNightTx',
    'Payment Format_enc', 'Currency_enc', 'port_number'
]

from sklearn.model_selection import train_test_split

edge_idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(edge_idx, test_size=0.30, stratify=df['Is Laundering'], random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.50, stratify=df.loc[temp_idx, 'Is Laundering'], random_state=42)
print(f"Train edges: {len(train_idx):,}  Val edges: {len(val_idx):,}  Test edges: {len(test_idx):,}")
print(f"Fraud — train: {df.loc[train_idx,'Is Laundering'].sum():.0f}  "
      f"val: {df.loc[val_idx,'Is Laundering'].sum():.0f}  test: {df.loc[test_idx,'Is Laundering'].sum():.0f}")

# ── Node features from TRAINING edges only (no leakage) ──
train_df = df.loc[train_idx]
node_feat = np.zeros((n_nodes, 6), dtype=np.float32)
out_deg = train_df.groupby('src')['Amount Paid'].agg(['count','sum','mean'])
in_deg  = train_df.groupby('dst')['Amount Received'].agg(['count','sum','mean'])
for idx, row in out_deg.iterrows():
    node_feat[idx, 0:3] = row.values
for idx, row in in_deg.iterrows():
    node_feat[idx, 3:6] = row.values
node_feat = np.log1p(node_feat)  # heavy-tailed -> log scale
print(f"✅ Node features: {node_feat.shape}")

## 🔁 Step 4 — Add reverse edges (the key GNN-specific adaptation)

Standard message passing only lets information flow in the edge's stored direction
(sender → receiver). Money laundering detection needs both directions — e.g. an account is
suspicious partly because of what it does with money *after* receiving it, which a
receiver-side-only view can't see. We add a reverse copy of every edge with a `is_reverse`
flag, so a normal PyG conv layer effectively becomes bidirectional while still knowing which
direction was the real transaction.

In [ ]:
import torch
from torch_geometric.data import Data

scaler_mean = train_df[EDGE_FEATURE_COLS].mean().values
scaler_std  = train_df[EDGE_FEATURE_COLS].std().replace(0, 1).values

def scale_edges(sub_df):
    X = (sub_df[EDGE_FEATURE_COLS].values - scaler_mean) / scaler_std
    return X.astype(np.float32)

edge_feat_all = scale_edges(df)

src = df['src'].values
dst = df['dst'].values

# Forward edges (flag=0) + reverse edges (flag=1), doubling the edge list
edge_index_fwd = np.stack([src, dst])
edge_index_rev = np.stack([dst, src])
edge_index = np.concatenate([edge_index_fwd, edge_index_rev], axis=1)

flag_fwd = np.zeros((len(df), 1), dtype=np.float32)
flag_rev = np.ones((len(df), 1), dtype=np.float32)
edge_attr = np.concatenate([
    np.concatenate([edge_feat_all, flag_fwd], axis=1),
    np.concatenate([edge_feat_all, flag_rev], axis=1),
], axis=0)

y_all = df['Is Laundering'].values.astype(np.float32)
# label/masks only apply to the forward half (reverse copies are structural only, not
# separately-labeled "transactions")
n_edges = len(df)
train_mask = np.zeros(2*n_edges, dtype=bool); train_mask[train_idx] = True
val_mask   = np.zeros(2*n_edges, dtype=bool); val_mask[val_idx]     = True
test_mask  = np.zeros(2*n_edges, dtype=bool); test_mask[test_idx]   = True

data = Data(
    x=torch.tensor(node_feat, dtype=torch.float32),
    edge_index=torch.tensor(edge_index, dtype=torch.long),
    edge_attr=torch.tensor(edge_attr, dtype=torch.float32),
    y=torch.tensor(np.concatenate([y_all, y_all]), dtype=torch.float32),
    train_mask=torch.tensor(train_mask),
    val_mask=torch.tensor(val_mask),
    test_mask=torch.tensor(test_mask),
)
print(data)
print(f"Edge feature dim: {data.edge_attr.shape[1]} (10 transaction features + port_number + is_reverse flag)")

## 🧠 Step 5 — Model: GINE-based Multi-GNN with edge classification head

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv

class MultiGNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden=64, n_layers=3, dropout=0.2):
        super().__init__()
        self.node_proj = nn.Linear(node_in, hidden)
        self.edge_proj = nn.Linear(edge_in, hidden)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.norms.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout

        # Edge classifier: [h_src, h_dst, edge_embedding] -> logit
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.node_proj(x))
        e = F.relu(self.edge_proj(edge_attr))
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, e)
            h_new = norm(h_new)
            h = F.relu(h_new) + h  # residual
            h = F.dropout(h, p=self.dropout, training=self.training)

        src, dst = edge_index
        edge_logits = self.edge_mlp(torch.cat([h[src], h[dst], e], dim=1))
        return edge_logits.squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiGNN(node_in=data.x.shape[1], edge_in=data.edge_attr.shape[1]).to(device)
data = data.to(device)
print(model)
print(f"Device: {device}")

## 🏋️ Step 6 — Train

Full-batch (the whole graph fits in memory for `HI-Small` on a T4 — a few hundred thousand
nodes, a few million edges). **If you get a CUDA out-of-memory error**: reduce `hidden` to 32,
or switch to PyG's `NeighborLoader` for mini-batch training (a bigger change — ask for help if
you need to go this route).

Loss uses `pos_weight` the same way we used `scale_pos_weight` for XGBoost, and — same
discipline as every other notebook here — the decision threshold is chosen on the **validation**
edges only, evaluated on test **once**.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score, classification_report, confusion_matrix

n_pos = data.y[data.train_mask].sum()
n_neg = data.train_mask.sum() - n_pos
pos_weight = torch.tensor([(n_neg / n_pos).item()], device=device)
print(f"pos_weight = {pos_weight.item():.1f}")

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val_ap = 0.0
best_state = None
patience, patience_ctr = 15, 0
EPOCHS = 100

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        val_scores = torch.sigmoid(logits[data.val_mask]).cpu().numpy()
        val_y = data.y[data.val_mask].cpu().numpy()
        val_ap = average_precision_score(val_y, val_scores)

    if val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  loss={loss.item():.4f}  val_AP={val_ap:.4f}  (best={best_val_ap:.4f})")

    if patience_ctr >= patience:
        print(f"Early stopping at epoch {epoch} (no val_AP improvement for {patience} epochs)")
        break

model.load_state_dict(best_state)
print(f"\n✅ Best validation AP: {best_val_ap:.4f}")

## 📊 Step 7 — Final evaluation (test set, touched once)

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_attr)
    all_scores = torch.sigmoid(logits).cpu().numpy()
    all_y = data.y.cpu().numpy()

val_scores  = all_scores[data.val_mask.cpu().numpy()]
val_y       = all_y[data.val_mask.cpu().numpy()]
test_scores = all_scores[data.test_mask.cpu().numpy()]
test_y      = all_y[data.test_mask.cpu().numpy()]

# threshold chosen on VALIDATION only
prec_v, rec_v, thr_v = precision_recall_curve(val_y, val_scores)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
best_thr = thr_v[np.argmax(f1_v)] if len(f1_v) else 0.5
print(f"Validation-chosen threshold: {best_thr:.4f}  (val F1={f1_v.max():.4f})")

test_roc = roc_auc_score(test_y, test_scores)
test_ap  = average_precision_score(test_y, test_scores)
test_pred = (test_scores >= best_thr).astype(int)
test_f1  = f1_score(test_y, test_pred)

print(f"\n=== GNN — FINAL TEST RESULTS ===")
print(f"ROC-AUC={test_roc:.4f}  AP={test_ap:.4f}  F1={test_f1:.4f}")
print(classification_report(test_y, test_pred, target_names=['Normal','Fraud'], digits=4))
print(confusion_matrix(test_y, test_pred))

print(f"\n📊 Comparison:")
print(f"  XGBoost (Notebook 3):  ROC-AUC=0.9838  AP=0.5766  F1=0.6207")
print(f"  GNN (this notebook):   ROC-AUC={test_roc:.4f}  AP={test_ap:.4f}  F1={test_f1:.4f}")
print(f"  Published Multi-PNA+EU on this data family: ROC-AUC~0.982-0.986  AP=0.672  F1=0.709")

## 💾 Step 8 — Save the model

If the GNN beats XGBoost, use it as your headline result and keep XGBoost as a documented
baseline/comparison in your report. If it doesn't (quite possible on a first pass — see the
honesty note in Step 9), keep XGBoost as the deployed model and report the GNN attempt with its
real numbers; a well-explained negative result is still a legitimate FYP contribution.

In [ ]:
import os, json
MODEL_PATH = '/content/saved_models'
os.makedirs(MODEL_PATH, exist_ok=True)

torch.save(model.state_dict(), f'{MODEL_PATH}/gnn_model.pt')
np.save(f'{MODEL_PATH}/gnn_scaler_mean.npy', scaler_mean)
np.save(f'{MODEL_PATH}/gnn_scaler_std.npy', scaler_std)

gnn_metadata = {
    'architecture': 'GINE-based Multi-GNN with reverse message passing + port numbering',
    'edge_feature_cols': EDGE_FEATURE_COLS + ['is_reverse'],
    'hidden_dim': 64, 'n_layers': 3,
    'threshold': float(best_thr),
    'test_performance': {'roc_auc': float(test_roc), 'avg_prec': float(test_ap), 'f1': float(test_f1)},
    'comparison': {
        'xgboost_roc_auc': 0.9838, 'xgboost_avg_prec': 0.5766, 'xgboost_f1': 0.6207,
        'published_multipna_roc_auc_range': [0.982, 0.986],
        'published_multipna_avg_prec': 0.672, 'published_multipna_f1': 0.709,
    }
}
with open(f'{MODEL_PATH}/gnn_metadata.json', 'w') as f:
    json.dump(gnn_metadata, f, indent=2)

print("✅ Saved gnn_model.pt, gnn_metadata.json")
print(json.dumps(gnn_metadata['test_performance'], indent=2))

import shutil
shutil.make_archive('/content/deepguard_gnn_model', 'zip', MODEL_PATH)
try:
    from google.colab import files
    files.download('/content/deepguard_gnn_model.zip')
except ImportError:
    print("ℹ️  Not in Colab — find the zip at /content/deepguard_gnn_model.zip")

## ⚠️ Step 9 — Reading your results honestly

A few things that can happen on a first run, and what they'd mean:

- **GNN beats XGBoost (F1 > 0.62, AP > 0.58):** Great — matches the published pattern. Use this
  as your primary model, cite the ~10–15 point AP/F1 gap over tabular XGBoost as your key
  finding, and explain *why* (graph structure captures multi-hop laundering patterns a
  per-transaction model can't see).
- **GNN is close to or slightly below XGBoost:** Also a legitimate, explainable result — not a
  failure. The published paper's full gain required more adaptations than reverse-MP + port
  numbering alone (see the paper's ablation table); this notebook implements the two adaptations
  shown to matter most, not the complete architecture. Report both numbers, explain the gap
  honestly, and you still have the stronger deployed model (XGBoost) either way.
- **GNN is dramatically worse (F1 near 0, or predicts everything as one class):** Usually means
  training instability, not a fundamentally broken idea. Try: a smaller learning rate (1e-4), a
  smaller `hidden` dim, or fewer layers (2 instead of 3) before concluding it doesn't work.
- **Training or inference crashes with an OOM error:** The full graph didn't fit on the GPU.
  Reduce `hidden` to 32, or come back for a mini-batch (`NeighborLoader`) version.

Whatever happens, **do not report a number you haven't actually seen come out of this notebook.**
If something looks off, paste the output back and we'll debug it together — that's a much
stronger position for your defense than a number you can't explain if asked how it was produced.